In [32]:
!pip install -q pydicom

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
!pip install -q timm transformers scikit-learn

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
import timm
from transformers import BertConfig, BertModel
from sklearn.preprocessing import StandardScaler

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
CONFIG = {
    "expression_col": "tpm_unstranded",
    "top_n_genes": 500,

    "genomic_hidden_size": 256,
    "genomic_attn_heads": 8,
    "genomic_layers": 4,
    "genomic_intermediate": 512,
    "genomic_dropout": 0.35,

    "swin_model": "swin_tiny_patch4_window7_224",
    "img_size": 224,
    "swin_embed_dim": 256,

    "fusion_dim": 256,
    "fusion_dropout": 0.3,
    "num_classes": 2,
}

In [19]:
class GenomicBERT(nn.Module):
    def __init__(self, n_genes, config):
        super().__init__()
        H = config["genomic_hidden_size"]

        self.gene_proj = nn.Linear(1, H)
        self.pos_emb = nn.Embedding(n_genes + 1, H)
        self.norm = nn.LayerNorm(H)
        self.drop = nn.Dropout(config["genomic_dropout"])

        bert_cfg = BertConfig(
            hidden_size=H,
            num_attention_heads=config["genomic_attn_heads"],
            num_hidden_layers=config["genomic_layers"],
            intermediate_size=config["genomic_intermediate"],
            max_position_embeddings=n_genes + 2,
            hidden_dropout_prob=config["genomic_dropout"],
            attention_probs_dropout_prob=config["genomic_dropout"],
            vocab_size=1,
        )

        self.encoder = BertModel(bert_cfg, add_pooling_layer=False)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, H))

    def get_embedding(self, x):
        B, G = x.shape
        gene_emb = self.gene_proj(x.unsqueeze(-1))
        cls = self.cls_token.expand(B, -1, -1)

        seq = torch.cat([cls, gene_emb], dim=1)
        pos = torch.arange(G + 1, device=x.device).unsqueeze(0)

        seq = self.drop(self.norm(seq + self.pos_emb(pos)))

        mask = torch.ones(seq.shape[0], seq.shape[1], device=x.device)
        out = self.encoder(inputs_embeds=seq, attention_mask=mask)

        return out.last_hidden_state[:, 0, :]

In [20]:
class SwinEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        backbone = timm.create_model(config["swin_model"], pretrained=False, num_classes=0)
        self.backbone = backbone
        self.proj = nn.Linear(backbone.num_features, config["swin_embed_dim"])

    def forward(self, x):
        return self.proj(self.backbone(x))

In [23]:
class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate_g = nn.Linear(dim * 2, dim)
        self.gate_v = nn.Linear(dim * 2, dim)

    def forward(self, g, v):
        cat = torch.cat([g, v], dim=-1)
        alpha = torch.sigmoid(self.gate_g(cat))
        beta = torch.sigmoid(self.gate_v(cat))
        return alpha * g + beta * v

In [24]:
class SwinEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        backbone = timm.create_model(config["swin_model"], pretrained=False, num_classes=0)
        self.backbone = backbone

        self.proj = nn.Sequential(   # ⚠️ MUST match
            nn.Linear(backbone.num_features, config["swin_embed_dim"]),
            nn.LayerNorm(config["swin_embed_dim"]),
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.proj(feats)

In [25]:
class MultimodalFusionModel(nn.Module):
    def __init__(self, n_genes, config):
        super().__init__()

        self.genomic_encoder = GenomicBERT(n_genes, config)
        self.image_encoder = SwinEncoder(config)
        self.fusion = GatedFusion(config["fusion_dim"])

        self.classifier = nn.Sequential(
            nn.LayerNorm(config["fusion_dim"]),
            nn.Linear(config["fusion_dim"], 128),
            nn.GELU(),
            nn.Dropout(config["fusion_dropout"]),
            nn.Linear(128, config["num_classes"])
        )

    def forward(self, gene_x, image_x):
        g = self.genomic_encoder.get_embedding(gene_x)
        v = self.image_encoder(image_x)

        fused = self.fusion(g, v)
        return self.classifier(fused)

In [27]:
FUSION_PATH = "/content/drive/MyDrive/best_fusion_model.pt"

ckpt = torch.load(FUSION_PATH, map_location=device, weights_only=False)

gene_ids = ckpt["gene_ids"]
scaler = ckpt["scaler"]
n_genes = ckpt["n_genes"]

model = MultimodalFusionModel(n_genes, CONFIG).to(device)
model.load_state_dict(ckpt["model_state"], strict=False)
model.eval()

print("✅ Model loaded successfully (ignored extra genomic head)")

✅ Model loaded successfully (ignored extra genomic head)


In [28]:
def tsv_to_gene_vector(tsv_path):
    df = pd.read_csv(tsv_path, sep="\t", comment="#")

    df = df[~df["gene_id"].str.startswith("N_", na=False)]
    df = df[df["gene_id"] != "gene_id"]

    df = df.set_index("gene_id")

    expr = pd.to_numeric(df[CONFIG["expression_col"]], errors="coerce").fillna(0.0)
    expr = expr.reindex(gene_ids, fill_value=0.0)

    x = np.log1p(expr.values)
    x = scaler.transform([x])[0]

    return torch.tensor(x, dtype=torch.float32).unsqueeze(0)

In [29]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

def load_image(path):
    img = Image.open(path).convert("RGB")
    return transform(img).unsqueeze(0)

In [33]:
import pydicom
import cv2
import numpy as np
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

def load_dicom(path):
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array.astype(np.float32)

    # Normalize (simple min-max)
    img = (img - img.min()) / (img.max() - img.min())
    img = (img * 255).astype(np.uint8)

    # Resize to model size
    img = cv2.resize(img, (224, 224))

    # Convert to 3-channel
    img = np.stack([img]*3, axis=-1)

    img = transform(img).unsqueeze(0)
    return img

In [34]:
def predict(tsv_path, dicom_path):

    gene = tsv_to_gene_vector(tsv_path).to(device)
    img = load_dicom(dicom_path).to(device)

    with torch.no_grad():
        logits = model(gene, img)
        probs = torch.softmax(logits, dim=1)

    prob = probs[0][1].item()
    label = "Tumor" if prob > 0.5 else "Healthy"

    print("=================================")
    print(f"Prediction : {label}")
    print(f"Tumor Prob : {prob:.4f}")
    print(f"Healthy    : {1-prob:.4f}")
    print("=================================")

In [35]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/genomic_dataset.zip"
EXTRACT_PATH = "/content/genomic_data"

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ Extracted to:", EXTRACT_PATH)

✅ Extracted to: /content/genomic_data


In [36]:
import glob

tsv_files = glob.glob("/content/genomic_data/genomic data set/healthy/0d11c50a-8648-48ea-a107-e138a1d3e086/bf3ea4a0-bcd6-4e9d-acbb-3416f6ce53b7.rna_seq.augmented_star_gene_counts.tsv", recursive=True)

print("Total TSV files:", len(tsv_files))
print("Example file:", tsv_files[0])

Total TSV files: 1
Example file: /content/genomic_data/genomic data set/healthy/0d11c50a-8648-48ea-a107-e138a1d3e086/bf3ea4a0-bcd6-4e9d-acbb-3416f6ce53b7.rna_seq.augmented_star_gene_counts.tsv


In [39]:
def predict_multimodal(tsv_path, dicom_path):

    # --- Genomic ---
    gene = tsv_to_gene_vector(tsv_path).to(device)

    # --- DICOM ---
    img = load_dicom(dicom_path).to(device)

    # --- Model ---
    with torch.no_grad():
        logits = model(gene, img)
        probs = torch.softmax(logits, dim=1)

    prob = probs[0][1].item()
    label = "Tumor" if prob > 0.5 else "Healthy"

    print("=================================")
    print(f"Prediction          : {label}")
    print(f"Tumor Probability   : {prob:.4f}")
    print(f"Healthy Probability : {1-prob:.4f}")
    print("=================================")

In [40]:
predict_multimodal(
    tsv_files[0],   # genomic file
    "/content/drive/MyDrive/data_/cancer/image (1).dcm"
)

Prediction          : Tumor
Tumor Probability   : 0.5643
Healthy Probability : 0.4357


In [41]:
predict_multimodal(
    tsv_files[0],   # genomic file
    "/content/drive/MyDrive/data_/healthy/healthy (1).dcm"
)

Prediction          : Healthy
Tumor Probability   : 0.4891
Healthy Probability : 0.5109
